# CX Assist: Notebook 01: S3 Data Access and Dynamic Case 360

This notebook validates the reusable data layer for **any case ID** stored in the CX Assist datasets. It loads structured CSV files from Amazon S3 and assembles related customer, vehicle, contract, invoice, payment, service, and interaction records.


## 1. Configuration

The notebook expects a project-root `.env` file containing `AWS_PROFILE`, `AWS_REGION`, `S3_BUCKET`, and `S3_PREFIX`. AWS credentials remain in the local AWS CLI profile.


In [1]:
from __future__ import annotations

import io
import os
from dataclasses import dataclass
from typing import Any

import boto3
import pandas as pd
from botocore.exceptions import BotoCoreError, ClientError, ProfileNotFound
from dotenv import load_dotenv

load_dotenv()

AWS_PROFILE = os.getenv("AWS_PROFILE", "CXASSIST")
AWS_REGION = os.getenv("AWS_REGION", "ap-south-1")
S3_BUCKET = os.getenv("S3_BUCKET", "rahulcxassistdemo")
S3_PREFIX = os.getenv("S3_PREFIX", "cx-copilot/dev").strip("/")

print({
    "profile": AWS_PROFILE,
    "region": AWS_REGION,
    "bucket": S3_BUCKET,
    "prefix": S3_PREFIX,
})


{'profile': 'CXASSIST', 'region': 'ap-south-1', 'bucket': 'rahulcxassistdemo', 'prefix': 'cx-copilot/dev'}


## 2. Reusable S3 dataset loader


In [2]:
@dataclass(frozen=True)
class S3Settings:
    profile: str
    region: str
    bucket: str
    prefix: str


class S3DatasetLoader:
    def __init__(self, settings: S3Settings) -> None:
        self.settings = settings
        try:
            session = boto3.Session(
                profile_name=settings.profile,
                region_name=settings.region,
            )
            self.s3 = session.client("s3")
        except ProfileNotFound as exc:
            raise RuntimeError(
                f"AWS profile '{settings.profile}' was not found."
            ) from exc

    def key(self, relative_path: str) -> str:
        return f"{self.settings.prefix}/{relative_path.lstrip('/')}"

    def list_objects(self, subfolder: str = "") -> list[str]:
        prefix = self.key(subfolder).rstrip("/") + "/"
        paginator = self.s3.get_paginator("list_objects_v2")
        keys: list[str] = []
        try:
            for page in paginator.paginate(
                Bucket=self.settings.bucket, Prefix=prefix
            ):
                keys.extend(
                    obj["Key"] for obj in page.get("Contents", [])
                    if not obj["Key"].endswith("/")
                )
        except (ClientError, BotoCoreError) as exc:
            raise RuntimeError(f"Unable to list S3 prefix '{prefix}'.") from exc
        return keys

    def read_bytes(self, relative_path: str) -> bytes:
        key = self.key(relative_path)
        try:
            response = self.s3.get_object(
                Bucket=self.settings.bucket, Key=key
            )
            return response["Body"].read()
        except ClientError as exc:
            raise RuntimeError(f"Unable to read S3 object '{key}'.") from exc

    def read_csv(self, filename: str) -> pd.DataFrame:
        content = self.read_bytes(f"structured/{filename}")
        return pd.read_csv(io.BytesIO(content))

    def read_policy(self, filename: str) -> str:
        return self.read_bytes(f"policies/{filename}").decode("utf-8")


settings = S3Settings(
    profile=AWS_PROFILE, region=AWS_REGION,
    bucket=S3_BUCKET, prefix=S3_PREFIX,
)
loader = S3DatasetLoader(settings)
loader.list_objects()[:5]


['cx-copilot/dev/README.md',
 'cx-copilot/dev/policies/billing_adjustment_policy.md',
 'cx-copilot/dev/policies/duplicate_charge_policy.md',
 'cx-copilot/dev/policies/escalation_and_approval_policy.md',
 'cx-copilot/dev/policies/maintenance_downtime_policy.md']

## 3. Load and inspect all structured datasets


In [3]:
DATASET_FILES = {
    "customers": "customers.csv",
    "vehicles": "vehicles.csv",
    "contracts": "contracts.csv",
    "invoices": "invoices.csv",
    "payments": "payments.csv",
    "service_records": "service_records.csv",
    "cases": "cases.csv",
    "case_interactions": "case_interactions.csv",
}

datasets = {
    name: loader.read_csv(filename)
    for name, filename in DATASET_FILES.items()
}

pd.DataFrame([
    {"dataset": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in datasets.items()
])


,dataset,rows,columns
0,customers,5,9
1,vehicles,5,10
2,contracts,5,9
3,invoices,6,12
4,payments,5,8
5,service_records,5,10
6,cases,5,11
7,case_interactions,7,7


## 4. Build a dynamic Case 360 context

No case ID is hard-coded inside the service. The selected case supplies the identifiers used for every related lookup.


In [4]:
class CaseNotFoundError(ValueError):
    pass


def records(frame: pd.DataFrame, column: str, value: Any) -> list[dict]:
    if pd.isna(value):
        return []
    result = frame.loc[frame[column] == value]
    return result.where(pd.notna(result), None).to_dict(orient="records")


def one_record(frame: pd.DataFrame, column: str, value: Any) -> dict | None:
    matches = records(frame, column, value)
    return matches[0] if matches else None


def get_case_context(case_id: str, data: dict[str, pd.DataFrame]) -> dict:
    normalized_id = case_id.strip().upper()
    case = one_record(data["cases"], "case_id", normalized_id)
    if case is None:
        available = sorted(data["cases"]["case_id"].tolist())
        raise CaseNotFoundError(
            f"Case '{normalized_id}' was not found. Available cases: {available}"
        )

    customer_id = case.get("customer_id")
    unit_number = case.get("unit_number")
    invoice_id = case.get("invoice_id")

    invoice = one_record(data["invoices"], "invoice_id", invoice_id)
    contract_id = invoice.get("contract_id") if invoice else None

    return {
        "case": case,
        "customer": one_record(data["customers"], "customer_id", customer_id),
        "vehicle": one_record(data["vehicles"], "unit_number", unit_number),
        "contract": one_record(data["contracts"], "contract_id", contract_id),
        "invoice": invoice,
        "payments": records(data["payments"], "invoice_id", invoice_id),
        "service_records": records(data["service_records"], "unit_number", unit_number),
        "interactions": records(data["case_interactions"], "case_id", normalized_id),
    }


## 5. Select any available case and inspect its context


In [5]:
available_case_ids = sorted(datasets["cases"]["case_id"].tolist())
print("Available cases:", available_case_ids)

selected_case_id = available_case_ids[0]  # Change this to any listed case ID.
case_context = get_case_context(selected_case_id, datasets)

print(f"Selected: {selected_case_id}")
print(f"Issue type: {case_context['case']['issue_type']}")
print(f"Customer: {case_context['customer']['customer_name']}")
print(f"Payments found: {len(case_context['payments'])}")
print(f"Service records found: {len(case_context['service_records'])}")
print(f"Interactions found: {len(case_context['interactions'])}")

case_context


Available cases: ['CASE-3021', 'CASE-3022', 'CASE-3023', 'CASE-3024', 'CASE-3025']
Selected: CASE-3021
Issue type: Billing Dispute
Customer: Northstar Retail Logistics
Payments found: 0
Service records found: 1
Interactions found: 2


{'case': {'case_id': 'CASE-3021',
  'customer_id': 'CUST-1001',
  'unit_number': 'UNIT-4521',
  'invoice_id': 'INV-1047',
  'opened_at': '2026-09-03T10:20:00Z',
  'channel': 'Email',
  'issue_type': 'Billing Dispute',
  'priority': 'High',
  'status': 'Open',
  'subject': 'Charge during vehicle downtime',
  'description': 'Customer disputes full monthly charge because vehicle was unavailable during an unscheduled repair.'},
 'customer': {'customer_id': 'CUST-1001',
  'customer_name': 'Northstar Retail Logistics',
  'segment': 'Enterprise',
  'primary_contact': 'Ava Carter',
  'email': 'ava.carter@example.test',
  'phone': '+1-555-0101',
  'city': 'Columbus',
  'state': 'OH',
  'account_status': 'Active'},
 'vehicle': {'unit_number': 'UNIT-4521',
  'customer_id': 'CUST-1001',
  'vin': 'DEMO1VIN000000001',
  'vehicle_type': 'Day Cab',
  'make': 'Freightliner',
  'model': 'Cascadia',
  'model_year': 2024,
  'operational_status': 'In Service',
  'current_location': 'Columbus OH',
  'odomet

## 6. Validate the service against every case


In [6]:
validation_results = []

for case_id in available_case_ids:
    context = get_case_context(case_id, datasets)
    validation_results.append({
        "case_id": case_id,
        "issue_type": context["case"]["issue_type"],
        "customer_found": context["customer"] is not None,
        "vehicle_found": context["vehicle"] is not None,
        "contract_found": context["contract"] is not None,
        "invoice_found": context["invoice"] is not None,
        "payment_count": len(context["payments"]),
        "service_count": len(context["service_records"]),
        "interaction_count": len(context["interactions"]),
    })

validation = pd.DataFrame(validation_results)
assert validation["customer_found"].all()
assert validation["vehicle_found"].all()
assert validation["contract_found"].all()
assert validation["invoice_found"].all()

validation


,case_id,issue_type,customer_found,vehicle_found,contract_found,invoice_found,payment_count,service_count,interaction_count
0,CASE-3021,Billing Dispute,True,True,True,True,0,1,2
1,CASE-3022,Duplicate Charge,True,True,True,True,1,1,1
2,CASE-3023,Billing Explanation,True,True,True,True,1,1,1
3,CASE-3024,Payment Reconciliation,True,True,True,True,1,1,1
4,CASE-3025,Roadside Assistance,True,True,True,True,1,1,2


## 7. Controlled missing-case test


In [7]:
try:
    get_case_context("CASE-9999", datasets)
except CaseNotFoundError as exc:
    print(f"PASS: {exc}")


PASS: Case 'CASE-9999' was not found. Available cases: ['CASE-3021', 'CASE-3022', 'CASE-3023', 'CASE-3024', 'CASE-3025']


## Completion criteria

Notebook 01 is complete when all datasets load, every available case produces a Case 360 context, and the unknown-case test produces a controlled error. The reusable classes and functions can then be moved into `src/cx_assist/data/` and `src/cx_assist/services/`.
